In [2]:
from dotenv import load_dotenv

load_dotenv()

True

## Creating subagents

In [3]:
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

@tool
def square(x: float) -> float:
    """Calculate the square of a number"""
    return x ** 2

In [14]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen3.5:4b")


In [15]:
from langchain.agents import create_agent

# create subagents

subagent_1 = create_agent(
    model=model,
    tools=[square_root]
)

subagent_2 = create_agent(
    model=model,
    tools=[square]
)

## Calling subagents

In [16]:
from langchain.messages import HumanMessage

@tool
def call_subagent_1(x: float) -> float:
    """Call subagent 1 in order to calculate the square root of a number"""
    response = subagent_1.invoke({"messages": [HumanMessage(content=f"Calculate the square root of {x}")]})
    return response["messages"][-1].content

@tool
def call_subagent_2(x: float) -> float:
    """Call subagent 2 in order to calculate the square of a number"""
    response = subagent_2.invoke({"messages": [HumanMessage(content=f"Calculate the square of {x}")]})
    return response["messages"][-1].content

## Creating the main agent

main_agent = create_agent(
    model=model,
    tools=[call_subagent_1, call_subagent_2],
    system_prompt="You are a helpful assistant who can call subagents to calculate the square root or square of a number.")

## Test

In [17]:
question = "What is the square root of 456?"

response = main_agent.invoke({"messages": [HumanMessage(content=question)]})

In [18]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='What is the square root of 456?', additional_kwargs={}, response_metadata={}, id='457db1c9-5605-4101-a21b-d79f964a0bc2'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3.5:4b', 'created_at': '2026-04-20T16:29:15.043358Z', 'done': True, 'done_reason': 'stop', 'total_duration': 7733337583, 'load_duration': 135318041, 'prompt_eval_count': 381, 'prompt_eval_duration': 1877577041, 'eval_count': 96, 'eval_duration': 5668432712, 'logprobs': None, 'model_name': 'qwen3.5:4b', 'model_provider': 'ollama'}, id='lc_run--019dabb9-656b-70a3-9db1-210d3aaaabcf-0', tool_calls=[{'name': 'call_subagent_1', 'args': {'x': 456}, 'id': 'dfb8cc54-dad4-434a-8cd2-0d8fe5483408', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 381, 'output_tokens': 96, 'total_tokens': 477}),
              ToolMessage(content='The square root of 456.0 is approximately **21.35**.', name='call_subagent_1', id='3b7c6337-1d65-